# BEV-WAM-X Stage 3.1c `z_t` Latent-Distillation Fine-Tune (A100 / H100)
Select a Colab **A100** (or H100) GPU runtime. Warm-starts from `stage31c-v1-run/latest.ckpt` (step 12,000) and distills `bev_fine` + all-frame `z_t`-only perception decoding for 2,656 updates (4 full epochs).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os
import subprocess
import sys

PACK = Path('/content/drive/MyDrive/bev-wam-x/stage31b-v2-pack')
OVERLAY = Path('/content/drive/MyDrive/bev-wam-x/stage31c-maptr-overlay')
INIT_CKPT = Path('/content/drive/MyDrive/bev-wam-x/stage31c-v1-run/latest.ckpt')
RUN = Path('/content/drive/MyDrive/bev-wam-x/stage31c-zdistill-v1-run')
REPO = Path('/content/bev-wam-x')
BRANCH = 'feat/v3-stage3-prep'
BATCH = 8
ACCUMULATE = 2
CHECKPOINT_EVERY = 300
NUM_WORKERS = 4
MAX_UPDATES = 2656
SCHEDULE_UPDATES = 2656
WARMUP_UPDATES = 150
LR = 5e-5
Z_AUX_WEIGHT = 0.6
Z_DISTILL_WEIGHT = 0.8
AUTO_UNASSIGN = True

def run(*args, cwd=None):
    subprocess.run(list(map(str, args)), cwd=cwd, check=True)

if not REPO.exists():
    run('git', 'clone', '--single-branch', '--branch', BRANCH,
        'https://github.com/AuroNoxx/bev-wam-x.git', REPO)
else:
    run('git', 'pull', '--ff-only', 'origin', BRANCH, cwd=REPO)
run('git', 'rev-parse', 'HEAD', cwd=REPO)
run(sys.executable, '-m', 'pip', 'install', '-r',
    REPO / 'v3_world_model/requirements_v3.txt')
import torch, torchvision
assert torch.cuda.is_available() and not torch.version.hip
gpu_name = torch.cuda.get_device_name(0)
assert any(g in gpu_name for g in ('A100', 'H100')) and torch.cuda.is_bf16_supported(), f'Expected A100/H100, got: {gpu_name}'
assert INIT_CKPT.is_file(), f'Missing init checkpoint: {INIT_CKPT}'
print('GPU:', gpu_name, 'torch:', torch.__version__, 'tv:', torchvision.__version__)


In [ ]:
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from v3_world_model.pack_colab_v3 import read_manifest, sha256

source_manifest = PACK / 'manifest.json'
manifest = read_manifest(source_manifest, check_shards=False)
dataset = Path('/content') / ('stage31-' + sha256(source_manifest)[:16])
if not (dataset / 'manifest.json').is_file():
    run(sys.executable, '-m', 'v3_world_model.pack_colab_v3', '--unpack', source_manifest, '--out', dataset, cwd=REPO)
assert sha256(dataset / 'manifest.json') == sha256(source_manifest)

overlay_done = dataset / '.stage31c_overlay.ok'
ov_manifest = OVERLAY / 'overlay_manifest.json'
if not overlay_done.is_file() or overlay_done.read_text(encoding='utf-8') != sha256(ov_manifest):
    run(sys.executable, '-m', 'v3_world_model.build_maptr_targets', '--dataset', dataset, '--unpack-overlay', OVERLAY, cwd=REPO)
    overlay_done.write_text(sha256(ov_manifest), encoding='utf-8')
print(f"Verified Stage 3.1c dataset + MapTR overlay: {manifest['total_frames']:,} frames")


In [ ]:
flags = ['--dataset', dataset, '--out', RUN, '--arch', 'stage31c', '--balanced-maps', '--no-pretrained',
         '--z-aux-weight', Z_AUX_WEIGHT, '--z-distill-weight', Z_DISTILL_WEIGHT, '--z-all-frames',
         '--batch', BATCH, '--accumulate', ACCUMULATE, '--workers', NUM_WORKERS,
         '--checkpoint-every', CHECKPOINT_EVERY,
         '--max-updates', MAX_UPDATES, '--schedule-updates', SCHEDULE_UPDATES,
         '--warmup-updates', WARMUP_UPDATES, '--lr', LR,
         '--sdpa', 'cuda_flash', '--precision', 'bf16-mixed']
if (RUN / 'latest.ckpt').is_file():
    flags += ['--resume']
else:
    flags += ['--init-from', INIT_CKPT]
run(sys.executable, '-m', 'v3_world_model.train_stage31_colab', *flags, cwd=REPO)

result = json.loads((RUN / 'training-result.json').read_text(encoding='utf-8'))
assert result['architecture'] == 'stage31c'
assert result['actual_updates'] >= MAX_UPDATES and result['physical_batch'] == BATCH
assert result['gradient_accumulation'] == ACCUMULATE and result['clip_len'] == 4
assert result['pack_sha256'] == sha256(source_manifest)
for name in ('best.ckpt', 'best_z_det.ckpt', 'latest.ckpt'):
    stored = torch.load(RUN / name, map_location='cpu', weights_only=False)
    assert stored['task_state']['pack_sha256'] == result['pack_sha256']
    assert stored['task_state']['architecture'] == 'stage31c'
    assert stored['state_dict'] and stored['optimizer_states'] and stored['lr_schedulers']
    assert 0 < stored['global_step'] <= result['actual_updates']
    if name == 'latest.ckpt':
        assert stored['global_step'] == result['actual_updates']
    del stored
print(json.dumps(result, indent=2))
if AUTO_UNASSIGN:
    from google.colab import runtime
    runtime.unassign()
